# Hands-On Lab 1: Predictive AI & Machine Learning in Google Colab
### Predicting Natural Rubber Compound Tensile Strength from the Recipe — Vibe Coding Edition

## The Problem
A rubber products plant is developing a carbon-black-filled **natural rubber (NR)** compound. Every lab trial costs a day: weigh, mix on the two-roll mill, run the rheometer (MDR), press-cure, condition and test. The lab has **120 past trials**. Can a machine learning model learn from them which recipe gives the best tensile strength, before we mix the next batch?

**Fixed in every trial (phr = parts per hundred rubber):** NR 100 · ZnO 5 · stearic acid 2 · antioxidant 1  
**Varied (Inputs $X$):** Sulphur, CBS accelerator, N330 carbon black, cure temperature  
**Target ($y$):** Tensile strength (`Tensile_Strength_MPa`)  
*(Note: Other post-cure properties such as hardness, torque, and t90 are ignored in this lab to focus strictly on predicting tensile strength directly from the recipe).*  

### 💡 What is "Vibe Coding"?
You describe your engineering intent in plain language; an AI assistant writes the code; **you** check whether the result makes physical sense.


### Step 1: Load the Lab Trials

| Column | Meaning | Role |
| :--- | :--- | :--- |
| `Sulphur_phr` | Sulphur (curative), 0.5 – 3.0 phr | Input ($X$) |
| `CBS_phr` | CBS accelerator, 0.5 – 2.0 phr | Input ($X$) |
| `Carbon_Black_N330_phr` | Reinforcing filler, 0 – 60 phr | Input ($X$) |
| `Cure_Temp_C` | Rheometer / press cure temperature, 140 – 170 °C | Input ($X$) |
| `t90_min` | Time to 90 % cure (MDR) | Measured property (ignored) |
| `Delta_Torque_dNm` | MDR torque rise M<sub>H</sub> − M<sub>L</sub> | Measured property (ignored) |
| `Hardness_ShoreA` | Hardness of the cured sheet | Measured property (ignored) |
| **`Tensile_Strength_MPa`** | **Tensile strength of the cured dumbbells** | **Single Target ($y$)** |


> 💬 **Vibe Coding Prompt:**  
> *"Load data from https://raw.githubusercontent.com/dissabnd/ITUM2-IntroAI/refs/heads/main/data/rubber_compound_lab_data.csv with pandas. Print the number of trials, show the first 5 rows, and show the min, mean and max of every numeric column."*


### Step 2: Data Plot — Look Before You Model
Inspect how key recipe ingredients affect our single target property (`Tensile_Strength_MPa`).  
Left: tensile strength against sulphur loading, coloured by carbon black. Right: tensile strength against carbon black loading, coloured by sulphur.


> 💬 **Vibe Coding Prompt:**  
> *"Make two plots side by side to explore what drives tensile strength. Left: scatter plot of `Sulphur_phr` (x) vs `Tensile_Strength_MPa` (y), coloured by `Carbon_Black_N330_phr` with a colour bar. Right: scatter plot of `Carbon_Black_N330_phr` (x) vs `Tensile_Strength_MPa` (y), coloured by `Sulphur_phr` with a colour bar. Add axis labels with units, titles, and a grid."*


✍️ **Question 2:**  
1. Does tensile strength keep rising as sulphur increases, or is there an optimum peak? What happens physically to a rubber network when it is over-crosslinked?  
2. How does carbon black loading influence tensile strength?


### Step 3: Train/Test Split & Model Training
- **Inputs ($X$):** The 4 recipe and cure variables (`Sulphur_phr`, `CBS_phr`, `Carbon_Black_N330_phr`, `Cure_Temp_C`).
- **Target ($y$):** Only `Tensile_Strength_MPa`.
- **Split:** Train on 80% (96 trials), test on 20% (24 trials) unseen.
- **Models:** A **Linear Regression** (baseline straight-line model) and a **Random Forest** (non-linear ensemble).

> ⚠️ **Data Leakage & Ignoring Other Properties:** We ignore `t90_min`, `Delta_Torque_dNm`, and `Hardness_ShoreA`. These are measured *after* mixing and curing in the lab, so they are not available when formulating a compound. Using them would cause data leakage.


> 💬 **Vibe Coding Prompt:**  
> *"Split the data into inputs X (`Sulphur_phr`, `CBS_phr`, `Carbon_Black_N330_phr`, `Cure_Temp_C`) and target y (`Tensile_Strength_MPa`). Split 80/20 with train_test_split (random_state=42). Train a LinearRegression and a RandomForestRegressor (200 trees, min_samples_leaf=2, random_state=42). Print R² and mean absolute error (MAE) on the test set for both models, and draw predicted vs. measured tensile strength plots side by side with a y = x dashed line."*


✍️ **Question 3:** Which model predicts tensile strength more accurately on the test set? Why does Random Forest outperform Linear Regression when predicting tensile strength?


### Step 4: Can We Trust Those Scores?
24 test trials is a small sample. **5-fold cross-validation** tests every trial once and reports an average score and standard deviation across all folds.


> 💬 **Vibe Coding Prompt:**  
> *"Run 5-fold cross-validation (shuffle=True, random_state=0) on both models for predicting `Tensile_Strength_MPa` using R² as the scoring metric. Print the mean and standard deviation of R² for both Linear Regression and Random Forest."*


✍️ **Question 4:** How do the 5-fold cross-validated R² scores compare with the single test split scores from Step 3? Which metric is more dependable for reporting model performance?


### Step 5: Feature Importance — What Drives Tensile Strength?
Now that we have trained our Random Forest model for `Tensile_Strength_MPa`, let us determine which recipe inputs have the greatest impact on tensile strength.
*(Notice: Rather than predicting multiple properties, we examine the relative importance of the 4 recipe inputs on tensile strength alone).* 


> 💬 **Vibe Coding Prompt:**  
> *"Extract the feature importances from the trained RandomForestRegressor for `Tensile_Strength_MPa`. Plot them as a horizontal bar chart sorted from highest to lowest importance, showing the percentage contribution of each recipe input (`Carbon_Black_N330_phr`, `Sulphur_phr`, `CBS_phr`, `Cure_Temp_C`)."*


✍️ **Question 5:**  
1. Which recipe input has the highest feature importance for tensile strength? Which input is second?  
2. Why is cure temperature (`Cure_Temp_C`) ranked low for tensile strength, even though it heavily influences curing speed?


### Step 6: Use the Model as a Formulator, and Test Its Limits
**You are the compound development technologist.** Hold CBS at 1.0 phr, black at 40 phr and cure at 150 °C. Sweep sulphur from 0.5 to 3.0 phr and ask both models for the predicted tensile strength.


> 💬 **Vibe Coding Prompt:**  
> *"Create a grid of sulphur from 0.5 to 3.0 phr with CBS 1.0 phr, carbon black 40 phr and cure temperature 150 °C. Predict tensile strength with both models and plot both curves. Print the sulphur level where the Random Forest predicts the highest tensile strength."*


Now try your own recipe. Run it at least twice: once inside the trial range, and once with **sulphur = 5.0 phr** (far outside the 0.5 – 3.0 phr the lab has tested).


> 💬 **Vibe Coding Prompt:**  
> *"Predict the tensile strength of one new recipe with the Random Forest. Before predicting, check every input against the min and max of the training data and print a clear extrapolation warning if any input is outside that range."*


✍️ **Question 6:** At 5.0 phr sulphur the Random Forest returns nearly the same tensile strength as at 3.0 phr. Given the trend you saw in the sweep, what would you expect a real 5 phr compound to do? What does this teach us about using tree-based models outside their training range?

---
## 💡 Key Takeaways
- **Look before you model:** Tensile strength shows an **optimum** around 1.5 phr sulphur and increases with carbon black reinforcement.
- **Match the model to the physics:** Linear regression fails to capture the optimum curve (the rise and fall cancel out), while Random Forest captures the non-linear relationship.
- **Small test sets flatter:** Cross-validation provides a more reliable and honest measure of model generalization.
- **Feature importance reveals key drivers:** Carbon black loading (~65%) and sulphur level (~27%) are the primary drivers of tensile strength.
- **Models do not extrapolate:** Outside the tested range, tree models predict constant values (flat-line). A materials engineer must verify boundary conditions before releasing new recipes.
- **Predictive vs. generative AI:** Here we learned from numbers to predict a property. In Labs 2 and 3, generative AI reasons in language and needs the same checking.
